# Part C: Mage Iris pipeline (Colab)
Runs the project that is already in the Git repo: `load_iris → validate_iris → prepare_data → train_model → (save_results, evaluate_model)`.
Run top to bottom and keep the outputs visible (they are part of the submission).

## 0. Get the code from GitHub

In [ ]:
GITHUB_USER = "meenakshishibu16"          # <- your GitHub user name
REPO = "freshcart-pipelines"
!git clone https://github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git log --oneline | head

## 1. Install Mage in a Python 3.12 environment (same as in class)

In [ ]:
!pip install -q uv
!uv venv -q -p 3.12 /content/mage_env
!uv pip install -q --python /content/mage_env mage-ai scikit-learn
import os
os.environ['PATH'] = '/content/mage_env/bin:' + os.environ['PATH']
!mage --help | head -5

## C1. Run the pipeline with tests (include this output in the report)

In [ ]:
!ls iris_project iris_project/pipelines/iris_classifier
!mage run iris_project iris_classifier --test

In [ ]:
import pandas as pd
pd.read_csv('iris_predictions.csv').head(10)

## C2. Tests as a gate: max_depth = 1 … 5
Before each run `iris_predictions.csv` is deleted. The table is filled in from the real output of each run.

In [ ]:
import subprocess, json, re, os, pandas as pd

rows = []
for d in [1, 2, 3, 4, 5]:
    if os.path.exists('iris_predictions.csv'):
        os.remove('iris_predictions.csv')
    r = subprocess.run(['mage', 'run', 'iris_project', 'iris_classifier', '--test',
                        '--runtime-vars', json.dumps({'max_depth': d})],
                       capture_output=True, text=True)
    out = r.stdout + r.stderr
    acc = re.search(r'Test accuracy = ([0-9.]+)', out)
    rows.append({
        'max_depth': d,
        'test accuracy': float(acc.group(1)) if acc else None,
        'test_accuracy passed?': 'Model accuracy is below 80%' not in out,
        'iris_predictions.csv created?': os.path.exists('iris_predictions.csv'),
    })
pd.DataFrame(rows)

In [ ]:
# Show the failing run in full (max_depth = 1): the assertion message and that save_results never ran
if os.path.exists('iris_predictions.csv'): os.remove('iris_predictions.csv')
!mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 1}' 2>&1 | grep -E "max_depth|Test accuracy|tests passed|Assertion|Saved|Failed" 
print('csv exists:', os.path.exists('iris_predictions.csv'))

## C3. The evaluation branch: `metadata.yaml`, confusion matrix and metrics

In [ ]:
!cat iris_project/pipelines/iris_classifier/metadata.yaml

In [ ]:
# Normal run: the confusion matrix and the metrics table are printed by evaluate_model
!mage run iris_project iris_classifier --test 2>&1 | sed -n '/Confusion matrix/,$p'

### Open the pipeline in the Mage UI (screenshot the graph with the branch)

In [ ]:
import subprocess, time, urllib.request

mage_server = subprocess.Popen(['/content/mage_env/bin/mage', 'start', 'iris_project'],
                               stdout=open('mage.log', 'w'), stderr=subprocess.STDOUT)
for i in range(60):
    try:
        urllib.request.urlopen('http://localhost:6789'); print('Mage is running'); break
    except Exception:
        time.sleep(2)
else:
    print('Mage did not start - check mage.log')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import subprocess, time, re
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:6789'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)
link = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
print('Open Mage here:', link[0] if link else 'Not ready yet - run this cell again')
# In the UI: Pipelines -> iris_classifier -> the dependency tree shows train_model with TWO children.

## C4. My own block: `validate_iris` (transformer + @test, between load_iris and prepare_data)

In [ ]:
!cat iris_project/transformers/validate_iris.py
!mage run iris_project iris_classifier --test 2>&1 | grep -E "Missing values|tests passed"